In [ ]:
from transformers import BartTokenizer
from datasets import Dataset
from transformers import BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch
from scripts.Utils import TimexNorm_Utils
from scripts.Reader import obtain_dataset

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
utils = TimexNorm_Utils(tokenizer)

In [ ]:
datasets = obtain_dataset("TempEval3", "normalised")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results/TimeNormBart",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    num_train_epochs=3,
    predict_with_generate=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
    data_collator=data_collator,
    compute_metrics=utils.compute_metrics,
)

In [ ]:
trainer.train()

Step,Training Loss


d:\GeoTKG\GeoTKG\Lib\site-packages\transformers\modeling_utils.py:3854: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=3, training_loss=11.223803202311197, metrics={'train_runtime': 2.0238, 'train_samples_per_second': 4.447, 'train_steps_per_second': 1.482, 'total_flos': 342976757760.0, 'train_loss': 11.223803202311197, 'epoch': 3.0})

In [ ]:
trainer.save_model("./results/TimeNormBart")
tokenizer.save_pretrained("./results/TimeNormBart")

('./time_norm_bart\\tokenizer_config.json',
 './time_norm_bart\\special_tokens_map.json',
 './time_norm_bart\\vocab.json',
 './time_norm_bart\\merges.txt',
 './time_norm_bart\\added_tokens.json')

In [7]:
import torch
from transformers import BartForConditionalGeneration, BartTokenizer
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load
tokenizer = BartTokenizer.from_pretrained(os.path.join("time_norm_bart"))
model = BartForConditionalGeneration.from_pretrained(os.path.join("time_norm_bart")).to(device)
model.eval()

def normalize_time(expr: str, ref_date: str) -> str:
    # build your exact input string
    inp = f"normalize time: {ref_date} | {expr}"
    # tokenize
    encoded = tokenizer(
        inp,
        return_tensors="pt",
        truncation=True,
        padding="longest",
    ).to(device)

    # generate
    out_ids = model.generate(
        **encoded,
        max_length=32,
        num_beams=4,       # beam search
        early_stopping=True
    )

    # decode
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

# example
print(normalize_time("next Friday", "2025-07-29"))   # → "2025-08-07"


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


normalize time: 2025-07-29 | next Friday
